# Topic 5 — ML Fundamentals
### Theory → tiny example → experiment.

This is the conceptual core everything else builds on. The pipeline you'll repeat for every
algorithm from here on:

```text
Data
 ↓
Features
 ↓
Model
 ↓
Prediction
 ↓
Loss
 ↓
Optimization
 ↓
Updated model
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)

## 1. Types of learning

- **Supervised learning**: you have labeled data (input X, correct output y). Learn X → y.
  e.g. text → cyberbullying/not-cyberbullying.
- **Unsupervised learning**: no labels. Find structure on your own (clustering, dimensionality reduction).
- **Semi-supervised**: a small amount of labeled data + a lot of unlabeled data.
- **Reinforcement learning**: an agent learns by trial and error via rewards/penalties (not used in your NLP work, just awareness).

In [ ]:
# Supervised: X (features) paired with y (labels)
X_supervised = np.array([[1, 2], [2, 3], [3, 1]])
y_supervised = np.array([0, 1, 0])   # each row of X has a known correct label
print("Supervised — X:\n", X_supervised, "\ny:", y_supervised)

# Unsupervised: only X, no y — the algorithm must find structure itself
X_unsupervised = np.array([[1, 2], [2, 3], [8, 9], [9, 8]])
print("\nUnsupervised — X:\n", X_unsupervised, "\n(no labels given)")

## 2. Features, labels, parameters, hyperparameters

- **Feature**: an input variable (a column). e.g. `text_length`, `num_exclamations`.
- **Label / target**: what you're predicting (a column). e.g. `is_bullying`.
- **Parameters**: values the model *learns* from data (e.g. the weights `w` and bias `b`).
- **Hyperparameters**: values *you* choose before training (e.g. learning rate, number of trees, k in KNN).

In [ ]:
# A trivial linear model: y = w*x + b
# w and b are PARAMETERS — learned from data
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([3, 5, 7, 9, 11])   # y = 2x + 1

model = LinearRegression()   # here, fit_intercept is a HYPERPARAMETER we could tune
model.fit(X, y)

print("learned parameter w (coef_):", model.coef_)
print("learned parameter b (intercept_):", model.intercept_)
# The model FOUND w≈2 and b≈1 on its own -- that's what "training" means.

## 3. Model, prediction, loss, optimization, training, inference

- **Model**: a function with adjustable parameters, e.g. `f(x) = wx + b`.
- **Prediction**: the model's output for a given input.
- **Loss (cost) function**: a number measuring how wrong the predictions are.
- **Optimization**: the process of adjusting parameters to reduce the loss (e.g. gradient descent).
- **Training**: repeatedly predicting → measuring loss → optimizing, on the *training* data.
- **Inference**: using the finished, trained model to predict on *new* data.

In [ ]:
def predict(w, b, x):
    return w * x + b

def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

X_flat = X.flatten()

# Before training: a random guess
w, b = 0.0, 0.0
preds_before = predict(w, b, X_flat)
print("loss with untrained model:", mse_loss(y, preds_before))

# "Training" (simplified gradient descent, same idea as Topic 4)
lr = 0.01
for step in range(2000):
    preds = predict(w, b, X_flat)
    dw = -2 * np.mean(X_flat * (y - preds))
    db = -2 * np.mean(y - preds)
    w -= lr * dw
    b -= lr * db

print("loss after training:", mse_loss(y, predict(w, b, X_flat)))
print("learned w, b:", w, b)

# Inference: use the trained model on a brand-new input
new_x = 10
print(f"inference — predicted y for x={new_x}:", predict(w, b, new_x))

## 4. Generalization, underfitting, overfitting

- **Generalization**: how well the model performs on *new, unseen* data (the actual goal).
- **Underfitting**: the model is too simple — high error on both train AND test data.
- **Overfitting**: the model memorizes training data (including its noise) — low train error, high test error.

We'll fit polynomials of increasing degree to the same noisy data to see all three regimes.

In [ ]:
# Generate noisy data from a true underlying curve
X_true = np.linspace(0, 10, 30).reshape(-1, 1)
y_true_curve = 0.5 * X_true.flatten()**2 - 2 * X_true.flatten() + 5
y_noisy = y_true_curve + rng.normal(0, 8, size=X_true.shape[0])

# Split into train/test manually for now (Topic 6 covers this properly)
train_idx = rng.choice(len(X_true), size=20, replace=False)
test_idx = np.array([i for i in range(len(X_true)) if i not in train_idx])

X_train, y_train = X_true[train_idx], y_noisy[train_idx]
X_test, y_test = X_true[test_idx], y_noisy[test_idx]

degrees = [1, 2, 15]   # underfit, good fit, overfit
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, degree in zip(axes, degrees):
    poly_model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    poly_model.fit(X_train, y_train)

    train_pred = poly_model.predict(X_train)
    test_pred = poly_model.predict(X_test)
    train_mse = mse_loss(y_train, train_pred)
    test_mse = mse_loss(y_test, test_pred)

    x_line = np.linspace(0, 10, 200).reshape(-1, 1)
    ax.scatter(X_train, y_train, label="train", color="blue")
    ax.scatter(X_test, y_test, label="test", color="orange")
    ax.plot(x_line, poly_model.predict(x_line), color="black")
    ax.set_title(f"degree={degree}\ntrain_mse={train_mse:.1f}  test_mse={test_mse:.1f}")
    ax.legend()

plt.tight_layout()
plt.show()
print("degree=1  -> UNDERFIT (too simple, high error everywhere)")
print("degree=2  -> GOOD FIT (matches the true quadratic shape)")
print("degree=15 -> OVERFIT (wiggles to chase noise, terrible on test data)")

## 5. Bias vs variance (the tradeoff underlying under/overfitting)

- **Bias**: error from overly simplistic assumptions (underfitting). The model is consistently wrong in the same way.
- **Variance**: error from being overly sensitive to the specific training data (overfitting).
  Retrain on a different sample and predictions swing wildly.

We simulate this by training the same-degree models on *many different* random training sets.

In [ ]:
def bias_variance_demo(degree, n_experiments=30):
    x_line = np.linspace(0, 10, 50).reshape(-1, 1)
    all_preds = []
    for _ in range(n_experiments):
        idx = rng.choice(len(X_true), size=20, replace=False)
        Xi, yi = X_true[idx], y_noisy[idx] + rng.normal(0, 8, size=20)  # new noise each time
        m = make_pipeline(PolynomialFeatures(degree), LinearRegression())
        m.fit(Xi, yi)
        all_preds.append(m.predict(x_line))
    return x_line, np.array(all_preds)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, degree in zip(axes, [1, 2, 15]):
    x_line, all_preds = bias_variance_demo(degree)
    for p in all_preds:
        ax.plot(x_line, p, color="gray", alpha=0.3)
    ax.plot(x_line, all_preds.mean(axis=0), color="red", label="average prediction")
    ax.set_title(f"degree={degree}")
    ax.legend()
plt.tight_layout()
plt.show()
# degree=1:  all gray lines nearly overlap (low variance) but far from the true curve (high bias)
# degree=15: gray lines scatter wildly (high variance) -- small data changes = very different model

## 6. Data leakage

**Data leakage** = information from outside the training set (often the test set, or the future)
accidentally influences training, making performance look better than it really is in production.

Common causes: scaling/normalizing using statistics from the *whole* dataset before splitting,
duplicate rows split across train/test, or using a feature that wouldn't be available at prediction time.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_demo = rng.normal(50, 10, size=(100, 1))

# ❌ WRONG: fit the scaler on ALL data, including what will become the test set
scaler_wrong = StandardScaler().fit(X_demo)  # sees test data statistics too!
X_scaled_wrong = scaler_wrong.transform(X_demo)
X_train_wrong, X_test_wrong = X_scaled_wrong[:80], X_scaled_wrong[80:]

# ✅ RIGHT: split first, fit scaler ONLY on train, apply same transform to test
X_train_raw, X_test_raw = X_demo[:80], X_demo[80:]
scaler_right = StandardScaler().fit(X_train_raw)     # never sees test data
X_train_right = scaler_right.transform(X_train_raw)
X_test_right = scaler_right.transform(X_test_raw)

print("Rule: split first, THEN fit any preprocessing (scalers, vectorizers, etc.) on train only.")

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Change `degree` in the underfit/overfit cell to 5 and 25 — describe what happens.
# 2. Change n_experiments in bias_variance_demo to 100 and see if the pattern gets clearer.
# 3. Write one sentence (in a markdown cell below) in your own words defining:
#    - overfitting
#    - underfitting
#    - data leakage
#    using an example from YOUR cyberbullying dataset for each.

---
### Next up: **Topic 6 — Train/Test/Validation splits** (the proper way to do what we hand-rolled above).

Say "next" when you're ready.